In [0]:

from pyspark.sql import SparkSession
from pyspark.sql.functions import *
data = [
(101,"Arjun Reddy","Hyderabad","Cardiology",5000,1),
(102,"Sneha Kapoor","Delhi","Orthopedics",3000,2),
(103,"Rahul Sharma","Mumbai","Dermatology",1500,1),
(104,"Priya Nair","Bangalore","Cardiology",5000,2),
(105,"Vikram Singh","Chennai","Neurology",7000,1),
(106,"Ananya Das","Kolkata","Orthopedics",3000,3),
(107,"Karan Patel","Ahmedabad","Cardiology",5000,1),
(108,"Meera Iyer","Bangalore","Dermatology",1500,2)
]
columns = [
"visit_id",
"patient_name",
"city",
"department",
"consultation_fee",
"tests_count"
]
columns = ["visit_id","patient_name","city","department","consultation_fee","tests_count"]

df = spark.createDataFrame(data, columns)
display(df)

visit_id,patient_name,city,department,consultation_fee,tests_count
101,Arjun Reddy,Hyderabad,Cardiology,5000,1
102,Sneha Kapoor,Delhi,Orthopedics,3000,2
103,Rahul Sharma,Mumbai,Dermatology,1500,1
104,Priya Nair,Bangalore,Cardiology,5000,2
105,Vikram Singh,Chennai,Neurology,7000,1
106,Ananya Das,Kolkata,Orthopedics,3000,3
107,Karan Patel,Ahmedabad,Cardiology,5000,1
108,Meera Iyer,Bangalore,Dermatology,1500,2


In [0]:
df=df.withColumn("total_fee",col("consultation_fee")+col("tests_count")*500)
df.show()

+--------+------------+---------+-----------+----------------+-----------+---------+
|visit_id|patient_name|     city| department|consultation_fee|tests_count|total_fee|
+--------+------------+---------+-----------+----------------+-----------+---------+
|     101| Arjun Reddy|Hyderabad| Cardiology|            5000|          1|     5500|
|     102|Sneha Kapoor|    Delhi|Orthopedics|            3000|          2|     4000|
|     103|Rahul Sharma|   Mumbai|Dermatology|            1500|          1|     2000|
|     104|  Priya Nair|Bangalore| Cardiology|            5000|          2|     6000|
|     105|Vikram Singh|  Chennai|  Neurology|            7000|          1|     7500|
|     106|  Ananya Das|  Kolkata|Orthopedics|            3000|          3|     4500|
|     107| Karan Patel|Ahmedabad| Cardiology|            5000|          1|     5500|
|     108|  Meera Iyer|Bangalore|Dermatology|            1500|          2|     2500|
+--------+------------+---------+-----------+----------------+---

In [0]:
high_value=df.filter(col("total_fee")>5000)
high_value.show()

+--------+------------+---------+----------+----------------+-----------+---------+
|visit_id|patient_name|     city|department|consultation_fee|tests_count|total_fee|
+--------+------------+---------+----------+----------------+-----------+---------+
|     101| Arjun Reddy|Hyderabad|Cardiology|            5000|          1|     5500|
|     104|  Priya Nair|Bangalore|Cardiology|            5000|          2|     6000|
|     105|Vikram Singh|  Chennai| Neurology|            7000|          1|     7500|
|     107| Karan Patel|Ahmedabad|Cardiology|            5000|          1|     5500|
+--------+------------+---------+----------+----------------+-----------+---------+



In [0]:
agg_df = df.groupBy("department").agg(sum("total_fee").alias("total_revenue"), avg("total_fee").alias("avg_revenue"))
agg_df.show()

+-----------+-------------+-----------------+
| department|total_revenue|      avg_revenue|
+-----------+-------------+-----------------+
| Cardiology|        17000|5666.666666666667|
|Orthopedics|         8500|           4250.0|
|Dermatology|         4500|           2250.0|
|  Neurology|         7500|           7500.0|
+-----------+-------------+-----------------+



In [0]:
agg_df.orderBy(desc("total_revenue")).show()

+-----------+-------------+-----------------+
| department|total_revenue|      avg_revenue|
+-----------+-------------+-----------------+
| Cardiology|        17000|5666.666666666667|
|Orthopedics|         8500|           4250.0|
|  Neurology|         7500|           7500.0|
|Dermatology|         4500|           2250.0|
+-----------+-------------+-----------------+



In [0]:
df.createOrReplaceTempView("df")

In [0]:
%sql
SELECT * FROM df WHERE department="Cardiology";

visit_id,patient_name,city,department,consultation_fee,tests_count,total_fee
101,Arjun Reddy,Hyderabad,Cardiology,5000,1,5500
104,Priya Nair,Bangalore,Cardiology,5000,2,6000
107,Karan Patel,Ahmedabad,Cardiology,5000,1,5500


In [0]:
%sql
SELECT city, SUM(consultation_fee) AS revenue
FROM df
GROUP BY city;

city,revenue
Hyderabad,5000
Delhi,3000
Mumbai,1500
Bangalore,6500
Chennai,7000
Kolkata,3000
Ahmedabad,5000


In [0]:
%sql
SELECT * FROM df ORDER BY consultation_fee DESC LIMIT 3;

visit_id,patient_name,city,department,consultation_fee,tests_count,total_fee
105,Vikram Singh,Chennai,Neurology,7000,1,7500
101,Arjun Reddy,Hyderabad,Cardiology,5000,1,5500
104,Priya Nair,Bangalore,Cardiology,5000,2,6000


In [0]:
%sql
SELECT department, COUNT(*) FROM df GROUP BY department;

department,COUNT(*)
Cardiology,3
Orthopedics,2
Dermatology,2
Neurology,1


In [0]:
%sql
CREATE DATABASE IF NOT EXISTS hospital_db;

USE hospital_db;

In [0]:
%sql
CREATE OR REPLACE TABLE patient_delta_sql(
  visit_id INT,
  patient_name STRING,
  city STRING,
  department STRING,
  consultation_fee INT,
  tests_count INT
)USING DELTA

In [0]:
%sql
INSERT INTO patient_delta_sql VALUES
(101,"Arjun Reddy","Hyderabad","Cardiology",5000,1),
(102,"Sneha Kapoor","Delhi","Orthopedics",3000,2),
(103,"Rahul Sharma","Mumbai","Dermatology",1500,1),
(104,"Priya Nair","Bangalore","Cardiology",5000,2),
(105,"Vikram Singh","Chennai","Neurology",7000,1),
(106,"Ananya Das","Kolkata","Orthopedics",3000,3),
(107,"Karan Patel","Ahmedabad","Cardiology",5000,1),
(108,"Meera Iyer","Bangalore","Dermatology",1500,2);

num_affected_rows,num_inserted_rows
8,8


In [0]:
%sql
SELECT * FROM patient_delta_sql;

visit_id,patient_name,city,department,consultation_fee,tests_count
101,Arjun Reddy,Hyderabad,Cardiology,5000,1
102,Sneha Kapoor,Delhi,Orthopedics,3000,2
103,Rahul Sharma,Mumbai,Dermatology,1500,1
104,Priya Nair,Bangalore,Cardiology,5000,2
105,Vikram Singh,Chennai,Neurology,7000,1
106,Ananya Das,Kolkata,Orthopedics,3000,3
107,Karan Patel,Ahmedabad,Cardiology,5000,1
108,Meera Iyer,Bangalore,Dermatology,1500,2


In [0]:
%sql
INSERT INTO patient_delta_sql VALUES
(109,"New Patient","Chennai","Cardiology",4000,2);

num_affected_rows,num_inserted_rows
1,1


In [0]:
%sql
UPDATE patient_delta_sql 
SET consultation_fee = 6000
WHERE visit_id=101;

num_affected_rows
1


In [0]:
%sql
DELETE FROM patient_delta_sql WHERE visit_id=103;
    
SELECT * FROM patient_delta_sql;

visit_id,patient_name,city,department,consultation_fee,tests_count
102,Sneha Kapoor,Delhi,Orthopedics,3000,2
104,Priya Nair,Bangalore,Cardiology,5000,2
105,Vikram Singh,Chennai,Neurology,7000,1
106,Ananya Das,Kolkata,Orthopedics,3000,3
107,Karan Patel,Ahmedabad,Cardiology,5000,1
108,Meera Iyer,Bangalore,Dermatology,1500,2
101,Arjun Reddy,Hyderabad,Cardiology,6000,1
109,New Patient,Chennai,Cardiology,4000,2


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW updates AS
SELECT * FROM VALUES
(101,"Arjun Reddy","Hyderabad","Cardiology",8000,2),
(110,"New Patient","Delhi","Dermatology",2000,1)
AS updates(visit_id, patient_name, city, department, consultation_fee, tests_count);

In [0]:
%sql
MERGE INTO patient_delta_sql t
USING updates s 
ON t.visit_id = s.visit_id
WHEN MATCHED THEN 
UPDATE SET *
WHEN NOT MATCHED THEN 
INSERT *;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
2,1,0,1


In [0]:
%sql
SELECT * FROM patient_delta_sql;
    


visit_id,patient_name,city,department,consultation_fee,tests_count
102,Sneha Kapoor,Delhi,Orthopedics,3000,2
104,Priya Nair,Bangalore,Cardiology,5000,2
105,Vikram Singh,Chennai,Neurology,7000,1
106,Ananya Das,Kolkata,Orthopedics,3000,3
107,Karan Patel,Ahmedabad,Cardiology,5000,1
108,Meera Iyer,Bangalore,Dermatology,1500,2
109,New Patient,Chennai,Cardiology,4000,2
101,Arjun Reddy,Hyderabad,Cardiology,8000,2
110,New Patient,Delhi,Dermatology,2000,1


In [0]:
%sql
DESCRIBE HISTORY patient_delta_sql;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
8,2026-05-04T04:48:05.000Z,145534544004666,azuser6404_mml.local@karthikirisoutlook.onmicrosoft.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(3767134651267114),686f7f3f-1f27-42ea-9075-662cf88517fd,0504-040032-y5opsisj-v2n,7,SnapshotIsolation,false,"Map(numRemovedFiles -> 3, numRemovedBytes -> 8011, p25FileSize -> 3077, numDeletionVectorsRemoved -> 1, minFileSize -> 3077, numAddedFiles -> 1, maxFileSize -> 3077, p75FileSize -> 3077, p50FileSize -> 3077, numAddedBytes -> 3077)",null,Databricks-Runtime/18.1.x-photon-scala2.13
7,2026-05-04T04:48:04.000Z,145534544004666,azuser6404_mml.local@karthikirisoutlook.onmicrosoft.com,MERGE,"Map(predicate -> [""(visit_id#15739 = visit_id#15727)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> true, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(3767134651267114),686f7f3f-1f27-42ea-9075-662cf88517fd,0504-040032-y5opsisj-v2n,6,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 2, numTargetBytesAdded -> 4965, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 1, executionTimeMs -> 3399, materializeSourceTimeMs -> 300, numTargetRowsInserted -> 1, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1474, numTargetRowsUpdated -> 1, numOutputRows -> 2, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 2, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1569)",null,Databricks-Runtime/18.1.x-photon-scala2.13
6,2026-05-04T04:39:49.000Z,145534544004666,azuser6404_mml.local@karthikirisoutlook.onmicrosoft.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(3767134651267114),e1e98dad-b265-4e96-b009-77ce7f5f49c8,0504-040032-y5opsisj-v2n,5,SnapshotIsolation,false,"Map(numRemovedFiles -> 1, numRemovedBytes -> 3071, p25FileSize -> 3046, numDeletionVectorsRemoved -> 1, minFileSize -> 3046, numAddedFiles -> 1, maxFileSize -> 3046, p75FileSize -> 3046, p50FileSize -> 3046, numAddedBytes -> 3046)",null,Databricks-Runtime/18.1.x-photon-scala2.13
5,2026-05-04T04:39:48.000Z,145534544004666,azuser6404_mml.local@karthikirisoutlook.onmicrosoft.com,DELETE,"Map(predicate -> [""(visit_id#15267 = 103)""])",null,List(3767134651267114),e1e98dad-b265-4e96-b009-77ce7f5f49c8,0504-040032-y5opsisj-v2n,4,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 1326, numDeletionVectorsUpdated -> 0, numDeletedRows -> 1, scanTimeMs -> 978, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 347)",null,Databricks-Runtime/18.1.x-photon-scala2.13
4,2026-05-04T04:38:53.000Z,145534544004666,azuser6404_mml.local@karthikirisoutlook.onmicrosoft.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(3767134651267114),7e81107f-8ad8-4b61-9406-43d4ee5d23f4,0504-040032-y5opsisj-v2n,3,SnapshotIsolation,false,"Map(numRemovedFiles -> 3, numRemovedBytes -> 6495, p25FileSize -> 3071, numDeletionVectorsRemoved -> 1, minFileSize -> 3071, numAddedFiles -> 1, maxFileSize -> 3071, p75FileSize -> 3071, p50FileSize -> 3071, numAddedBytes -> 3071)",null,Databricks-Runtime/18.1.x-photon-scala2.13
3,2026-05-04T04:38:51.000Z,145534544004666,azuser6404_mml.local@karthikirisoutlook.onmicrosoft.com,UPDATE,"Map(predicate -> [""(visit_id#14623 = 101)""])",null,List(3767134651267114),7e81107f-8ad8-4b61-9406-43d4ee5d23f4,0504-04

In [0]:
%sql
SELECT * FROM patient_delta_sql VERSION AS OF 1;

visit_id,patient_name,city,department,consultation_fee,tests_count
101,Arjun Reddy,Hyderabad,Cardiology,5000,1
102,Sneha Kapoor,Delhi,Orthopedics,3000,2
103,Rahul Sharma,Mumbai,Dermatology,1500,1
104,Priya Nair,Bangalore,Cardiology,5000,2
105,Vikram Singh,Chennai,Neurology,7000,1
106,Ananya Das,Kolkata,Orthopedics,3000,3
107,Karan Patel,Ahmedabad,Cardiology,5000,1
108,Meera Iyer,Bangalore,Dermatology,1500,2


3. Explain the Effect of VACUUM

VACUUM in Delta Lake is used to remove old, unused data files from the table storage.

Effect:
Deletes obsolete files created after UPDATE, DELETE, MERGE
Frees up storage space
Improves query performance

In [0]:
%sql
VACUUM patient_delta_sql RETAIN 168 HOURS DRY RUN;

In [0]:
%sql
CREATE TABLE hospital_db.patient_parquet (
  visit_id INT,
  patient_name STRING,
  city STRING,
  department STRING,
  consultation_fee INT,
  tests_count INT
)
USING DELTA;

In [0]:
%sql
INSERT INTO hospital_db.patient_parquet
SELECT * FROM hospital_db.patient_delta_sql;

num_affected_rows,num_inserted_rows
9,9


In [0]:
%sql
CONVERT TO DELTA hospital_db.patient_parquet;

In [0]:
%sql
DESCRIBE DETAIL hospital_db.patient_parquet;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,72f334d2-03ee-4a0d-9782-1c2033d2ffea,hexa_compute_hm.hospital_db.patient_parquet,null,abfss://unity-catalog-storage@dbstorageacgje3hu6ta24.dfs.core.windows.net/7405606333390291/__unitystorage/catalogs/3b93abb9-c225-4820-bacd-0cb186da1085/tables/8caed36a-f3cd-4094-be65-65a567a53886,2026-05-04T05:21:58.987Z,2026-05-04T05:22:36.000Z,List(),List(),1,2246,"Map(delta.parquet.compression.codec -> zstd, delta.enableDeletionVectors -> true, delta.enableRowTracking -> true, delta.rowTracking.materializedRowCommitVersionColumnName -> _row-commit-version-col-50a8a83e-61f2-4710-b06f-c7ebb0a21b72, delta.rowTracking.materializedRowIdColumnName -> _row-id-col-444d4232-8cfe-4fad-ae9c-c5f62b421837)",3,7,"List(appendOnly, deletionVectors, domainMetadata, invariants, rowTracking)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
%sql
CREATE OR REPLACE TABLE hospital_db.patient_target (
  visit_id INT,
  patient_name STRING,
  city STRING,
  department STRING,
  consultation_fee INT,
  tests_count INT
)
USING DELTA;

In [0]:
%sql
INSERT INTO hospital_db.patient_target
SELECT * FROM hospital_db.patient_delta_sql;

num_affected_rows,num_inserted_rows
9,9


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW patient_updates AS
SELECT * FROM VALUES
(101,"Arjun Reddy","Hyderabad","Cardiology",8000,2), 
(110,"New Patient","Delhi","Dermatology",2000,1)     
AS patient_updates(visit_id, patient_name, city, department, consultation_fee, tests_count);

In [0]:
%sql
MERGE INTO hospital_db.patient_target AS t
USING patient_updates AS s
ON t.visit_id = s.visit_id
WHEN MATCHED THEN 
UPDATE SET
t.patient_name = s.patient_name,
t.city = s.city,
t.department = s.department,
t.consultation_fee = s.consultation_fee,
t.tests_count = s.tests_count
WHEN NOT MATCHED THEN
INSERT (
    visit_id, patient_name, city, department, consultation_fee, tests_count
  )
  VALUES (
    s.visit_id, s.patient_name, s.city, s.department, s.consultation_fee, s.tests_count
  );

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
2,2,0,0


In [0]:
%sql
SELECT * 
FROM hospital_db.patient_target
WHERE visit_id = 101;

visit_id,patient_name,city,department,consultation_fee,tests_count
101,Arjun Reddy,Hyderabad,Cardiology,8000,2


In [0]:
%sql
SELECT * 
FROM hospital_db.patient_target
WHERE visit_id = 110;

visit_id,patient_name,city,department,consultation_fee,tests_count
110,New Patient,Delhi,Dermatology,2000,1


In [0]:
%sql
SHOW CATALOGS;

catalog
assessment
bronze
datas
hexa_compute_hm
hexacatalog_student
hive_metastore
medallion
samples
sensor_catalog
system


In [0]:
%sql
USE CATALOG hexa_compute_hm;

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS hospital_schema;

In [0]:
%sql
USE hospital_schema;

In [0]:
%sql
CREATE TABLE patient_table(
  visit_id INT,
  patient_name STRING,
  city STRING,
  department STRING,
  consultation_fee INT,
  tests_count INT
)USING DELTA;

In [0]:
%sql
SHOW TABLES IN hospital_db;

database,tableName,isTemporary
hospital_db,patient_delta_sql,false
hospital_db,patient_parquet,false
hospital_db,patient_target,false
,df,true
,patient_updates,true
,updates,true


In [0]:
%sql
INSERT INTO patient_table
SELECT * FROM hospital_db.patient_delta_sql;

num_affected_rows,num_inserted_rows
9,9


In [0]:
%sql
SHOW TABLES IN hospital_db;

database,tableName,isTemporary
hospital_db,patient_delta_sql,false
hospital_db,patient_parquet,false
hospital_db,patient_target,false
,df,true
,patient_updates,true
,updates,true


In [0]:
%sql
CREATE OR REPLACE TABLE hospital_db.dept_summary AS
SELECT 
  department,
  COUNT(*) AS total_patients,
  SUM(consultation_fee) AS total_revenue
FROM patient_table
GROUP BY department;

num_affected_rows,num_inserted_rows


In [0]:
%sql
SHOW USERS;

name
azuser6404_mml.local@karthikirisoutlook.onmicrosoft.com
